In [13]:
import random
import time
from typing import Union
import matplotlib.pyplot as plt

class Faction:
    """
    Represents a faction in the 'apple wars', tracking fruit resources and allowing interactions
    like growth and attacks on other factions.
    """

    def __init__(self, name: str, apples: int, oranges: int, bananas: int, strawberries: int):
        """Initialize faction with name and fruit resources."""
        # Validate input types
        if not isinstance(name, str):
            raise TypeError("Faction name must be a string.")
        for fruit_name, amount in zip(
            ['apples', 'oranges', 'bananas', 'strawberries'],
            [apples, oranges, bananas, strawberries]
        ):
            if not isinstance(amount, int):
                raise TypeError(f"{fruit_name.capitalize()} amount must be an integer.")
            if not 0 <= amount <= 200:
                raise ValueError(f"{fruit_name.capitalize()} amount must be between 0 and 200.")

        # Private attributes
        self._name = name
        self._apples = apples
        self._oranges = oranges
        self._bananas = bananas
        self._strawberries = strawberries

    # ------------------------------
    # Getters and Setters
    # ------------------------------
    @property
    def name(self) -> str:
        return self._name

    @name.setter
    def name(self, value: str) -> None:
        if not isinstance(value, str):
            raise TypeError("Faction name must be a string.")
        self._name = value

    @property
    def apples(self) -> int:
        return self._apples

    @apples.setter
    def apples(self, value: int) -> None:
        self._validate_fruit_amount(value, 'apples')
        self._apples = value

    @property
    def oranges(self) -> int:
        return self._oranges

    @oranges.setter
    def oranges(self, value: int) -> None:
        self._validate_fruit_amount(value, 'oranges')
        self._oranges = value

    @property
    def bananas(self) -> int:
        return self._bananas

    @bananas.setter
    def bananas(self, value: int) -> None:
        self._validate_fruit_amount(value, 'bananas')
        self._bananas = value

    @property
    def strawberries(self) -> int:
        return self._strawberries

    @strawberries.setter
    def strawberries(self, value: int) -> None:
        self._validate_fruit_amount(value, 'strawberries')
        self._strawberries = value

    def _validate_fruit_amount(self, value: int, fruit_name: str) -> None:
        if not isinstance(value, int):
            raise TypeError(f"{fruit_name.capitalize()} amount must be an integer.")
        if not 0 <= value <= 200:
            raise ValueError(f"{fruit_name.capitalize()} amount must be between 0 and 200.")

    # ------------------------------
    # Representation
    # ------------------------------
    def __repr__(self) -> str:
        return (f"Faction(name='{self._name}', apples={self._apples}, "
                f"oranges={self._oranges}, bananas={self._bananas}, strawberries={self._strawberries})")

    # ------------------------------
    # Methods
    # ------------------------------
    def inventory(self) -> int:
        """Return total number of fruits."""
        return self._apples + self._oranges + self._bananas + self._strawberries

    def status(self) -> None:
        """Print a status message depending on the total number of fruits."""
        total = self.inventory()
        if total <= 150:
            print(f"This aint nothin but a scratch! {self._name} might be struggling for the moment, but will be back with vengeance!")
        elif 150 < total <= 500:
            print(f"The proud {self._name} is VERY eager to toss some apples! Where are the opponents?!")
        else:
            print(f"{self._name} is GODLIKE! Their position is super dominant!")

    def visualize_resources(self) -> None:
        """Plot a bar chart of the current fruit resources with colors by amount."""
        fruits = ['apples', 'oranges', 'bananas', 'strawberries']
        values = [self._apples, self._oranges, self._bananas, self._strawberries]
        colors = ['green' if v >= 100 else 'orange' if v > 50 else 'red' for v in values]

        plt.bar(fruits, values, color=colors)
        plt.ylim(0, 220)
        plt.title(f"{self._name}'s Fruit Inventory")
        plt.ylabel("Quantity")
        plt.show()

    def organical_growth(self) -> None:
        """Randomly grow fruits by 1-5 units without exceeding maximum 200."""
        for fruit in ['apples', 'oranges', 'bananas', 'strawberries']:
            current_amount = getattr(self, fruit)
            if current_amount < 200:
                growth = random.randint(1, 5)
                new_amount = min(200, current_amount + growth)
                setattr(self, fruit, new_amount)
                print(f"Glorious {self._name} has harvested {new_amount - current_amount} {fruit}, to a new total of {new_amount}!")
            else:
                print(f"Glorious {self._name} is already at maximum capacity for {fruit}!")

    
    def __mul__(self, other) -> None:
        """Simulate an attack on another faction with drama, critical hits, and loot mechanics."""
        import time
        import random

        print(f"⚔️  {self.name} is attacking {other.name}!")
        time.sleep(1.5)

        # Check if the defender has any resources
        if other.inventory() == 0:
            print(f"🚫 {other.name} has no resources left. Choose another target!")
            return

        # Determine if attack succeeds (70% chance)
        if random.random() > 0.7:
            print(f"🛡️  {other.name} skillfully defended the attack!")
            return

        print("🔥 The attack was successful!")
        time.sleep(1.5)

        # Check for critical hit (10% chance)
        critical = random.random() <= 0.1
        loot_percent = 0.4 if critical else 0.2
        if critical:
            print(f"💥 CRITICAL STRIKE! {self.name} will loot {int(loot_percent*100)}% of all resources!")
        else:
            print(f"💰 {self.name} will loot {int(loot_percent*100)}% of all resources!")
        time.sleep(1.5)

        # Process each fruit
        fruits = ['apples', 'oranges', 'bananas', 'strawberries']
        for fruit in fruits:
            defender_amount = getattr(other, fruit)
            attack_amount = getattr(self, fruit)

            # HUMILIATING BLOW if very few resources left
            if 0 < defender_amount <= 5:
                loot = defender_amount
                print(f"😱 HUMILIATING BLOW! {self.name} stole ALL {loot} {fruit} from {other.name}!")
            else:
                loot = int(defender_amount * loot_percent)
                if loot > 0:
                    print(f"🏴‍☠️  {self.name} prepares to loot {loot} {fruit}...")

            # Update defender
            setattr(other, fruit, max(defender_amount - loot, 0))

            # Update attacker (max 200)
            space_left = 200 - attack_amount
            gained = min(loot, space_left)
            setattr(self, fruit, attack_amount + gained)

            # Report chaos if overflow occurs
            if gained < loot:
                print(f"⚖️ {self.name} looted {gained} {fruit}, but {loot - gained} were lost in the chaos of war!")
            elif loot > 0:
                print(f"💰 {self.name} looted {gained} {fruit} from {other.name}!")

            time.sleep(1.2)

            print(f"✅ {self.name} finishes the attack on {other.name}!\n")

        else:
            print(f"{other.name} skillfully defended the attack!")


In [15]:
fire_nation = Faction('Fire Nation', 120, 120, 150, 180)
earth_kingdom = Faction('Earth Kingdom', 6, 4, 2, 1)
water_tribe = Faction('Water Tribe', 100, 100, 100, 100)
air_nomads = Faction('Air Nomads', 100, 100, 100, 100)

fire_nation * water_tribe

⚔️  Fire Nation is attacking Water Tribe!
🔥 The attack was successful!
💰 Fire Nation will loot 20% of all resources!
🏴‍☠️  Fire Nation prepares to loot 20 apples...
💰 Fire Nation looted 20 apples from Water Tribe!
✅ Fire Nation finishes the attack on Water Tribe!

🏴‍☠️  Fire Nation prepares to loot 20 oranges...
💰 Fire Nation looted 20 oranges from Water Tribe!
✅ Fire Nation finishes the attack on Water Tribe!

🏴‍☠️  Fire Nation prepares to loot 20 bananas...
💰 Fire Nation looted 20 bananas from Water Tribe!
✅ Fire Nation finishes the attack on Water Tribe!

🏴‍☠️  Fire Nation prepares to loot 20 strawberries...
💰 Fire Nation looted 20 strawberries from Water Tribe!
✅ Fire Nation finishes the attack on Water Tribe!

Water Tribe skillfully defended the attack!
